In [2]:
import pandas as pd
import numpy as np

# Read delivery dataset
delivery_data = pd.read_csv('cleaned_delivery_data.csv')

# Read pickup dataset
pickup_data = pd.read_csv('cleaned_pickup_data.csv')

# Display first 5 rows
print(delivery_data.head())
print(pickup_data.head())

   order_id  region_id  city  courier_id        lng       lat  aoi_id  \
0   2071340         86     4        3135  121.40538  37.35796     175   
1   3767291         86     4        3135  121.40539  37.35804     175   
2   2184812         86     4        3135  121.40533  37.35809     175   
3   4048066         86     4        1586  121.51762  37.45354     293   
4   2309133         86     4        1508  121.40041  37.41166     314   

   aoi_type  accept_time  accept_gps_time  accept_gps_lng  accept_gps_lat  \
0        14          NaN              NaN       121.38013        37.41137   
1        14          NaN              NaN       121.38042        37.41050   
2        14          NaN              NaN       121.39184        37.41000   
3         1          NaN              NaN       121.38107        37.41032   
4        14          NaN              NaN       121.38042        37.41083   

   delivery_time  delivery_gps_time  delivery_gps_lng  delivery_gps_lat   ds  \
0            NaN  

In [3]:
# merge the 2 dataset into one
merged_data = pd.concat([delivery_data, pickup_data], ignore_index=True)
#show the merged the dataset
print(merged_data.head())


   order_id  region_id  city  courier_id        lng       lat  aoi_id  \
0   2071340         86     4        3135  121.40538  37.35796     175   
1   3767291         86     4        3135  121.40539  37.35804     175   
2   2184812         86     4        3135  121.40533  37.35809     175   
3   4048066         86     4        1586  121.51762  37.45354     293   
4   2309133         86     4        1508  121.40041  37.41166     314   

   aoi_type  accept_time  accept_gps_time  ...   ds  hour  weekday  \
0        14          NaN              NaN  ...  618   NaN      NaN   
1        14          NaN              NaN  ...  814   NaN      NaN   
2        14          NaN              NaN  ...  901   NaN      NaN   
3         1          NaN              NaN  ...  825   NaN      NaN   
4        14          NaN              NaN  ...  709   NaN      NaN   

   is_weekend  time_window_start  time_window_end  pickup_time  \
0           0                NaN              NaN          NaN   
1       

In [4]:
print(merged_data.shape)

(481415, 26)


In [5]:
print(merged_data.isnull().sum())

order_id                  0
region_id                 0
city                      0
courier_id                0
lng                       0
lat                       0
aoi_id                    0
aoi_type                  0
accept_time          481415
accept_gps_time      481415
accept_gps_lng       100947
accept_gps_lat       100947
delivery_time        481415
delivery_gps_time    481415
delivery_gps_lng     250000
delivery_gps_lat     250000
ds                        0
hour                 481415
weekday              481415
is_weekend                0
time_window_start    481415
time_window_end      481415
pickup_time          481415
pickup_gps_time      481415
pickup_gps_lng       280641
pickup_gps_lat       280641
dtype: int64


In [6]:
print('Original shape of merged_data:', merged_data.shape)

# Get columns with all null values
all_null_columns = merged_data.columns[merged_data.isnull().all()].tolist()
print(f'Columns with all null values: {all_null_columns}')

# Drop columns that are entirely null
merged_data = merged_data.drop(columns=all_null_columns)

print('Shape of merged_data after dropping all-null columns:', merged_data.shape)

# Display null counts again for remaining columns
print('\nNull counts after dropping all-null columns:')
print(merged_data.isnull().sum())

Original shape of merged_data: (481415, 26)
Columns with all null values: ['accept_time', 'accept_gps_time', 'delivery_time', 'delivery_gps_time', 'hour', 'weekday', 'time_window_start', 'time_window_end', 'pickup_time', 'pickup_gps_time']
Shape of merged_data after dropping all-null columns: (481415, 16)

Null counts after dropping all-null columns:
order_id                 0
region_id                0
city                     0
courier_id               0
lng                      0
lat                      0
aoi_id                   0
aoi_type                 0
accept_gps_lng      100947
accept_gps_lat      100947
delivery_gps_lng    250000
delivery_gps_lat    250000
ds                       0
is_weekend               0
pickup_gps_lng      280641
pickup_gps_lat      280641
dtype: int64


In [7]:
print('Shape of merged_data before dropping rows with nulls:', merged_data.shape)

# Drop rows that contain any null values
merged_data.dropna(inplace=True)

print('Shape of merged_data after dropping rows with nulls:', merged_data.shape)

# Verify that there are no more null values
print('\nNull counts after dropping rows with any null values:')
print(merged_data.isnull().sum())

Shape of merged_data before dropping rows with nulls: (481415, 16)
Shape of merged_data after dropping rows with nulls: (0, 16)

Null counts after dropping rows with any null values:
order_id            0
region_id           0
city                0
courier_id          0
lng                 0
lat                 0
aoi_id              0
aoi_type            0
accept_gps_lng      0
accept_gps_lat      0
delivery_gps_lng    0
delivery_gps_lat    0
ds                  0
is_weekend          0
pickup_gps_lng      0
pickup_gps_lat      0
dtype: int64


### Re-creating `merged_data` and imputing missing values

Since our previous attempt to drop rows with null values resulted in an empty DataFrame, we'll re-load the original data and try a different strategy: imputing missing values.

First, we'll re-load the `delivery_data` and `pickup_data` and merge them. Then, we'll drop columns that are *completely* null (as they have no information to impute). For the remaining columns with partial missing values (the `_gps_lng` and `_gps_lat` columns), we will fill them using the mean value of their respective columns.

In [8]:
import pandas as pd

# Re-load delivery dataset (assuming cleaned_delivery_data.csv is available)
delivery_data = pd.read_csv('cleaned_delivery_data.csv')

# Re-load pickup dataset (assuming cleaned_pickup_data.csv is available)
pickup_data = pd.read_csv('cleaned_pickup_data.csv')

# Re-merge the two datasets into one
merged_data = pd.concat([delivery_data, pickup_data], ignore_index=True)

print('Shape of newly merged_data:', merged_data.shape)
print('\nNull counts of newly merged_data before any processing:')
print(merged_data.isnull().sum())

Shape of newly merged_data: (481415, 26)

Null counts of newly merged_data before any processing:
order_id                  0
region_id                 0
city                      0
courier_id                0
lng                       0
lat                       0
aoi_id                    0
aoi_type                  0
accept_time          481415
accept_gps_time      481415
accept_gps_lng       100947
accept_gps_lat       100947
delivery_time        481415
delivery_gps_time    481415
delivery_gps_lng     250000
delivery_gps_lat     250000
ds                        0
hour                 481415
weekday              481415
is_weekend                0
time_window_start    481415
time_window_end      481415
pickup_time          481415
pickup_gps_time      481415
pickup_gps_lng       280641
pickup_gps_lat       280641
dtype: int64


In [9]:
# Drop columns that are entirely null (as identified before)
all_null_columns = merged_data.columns[merged_data.isnull().all()].tolist()
print(f'Columns with all null values to be dropped: {all_null_columns}')
merged_data = merged_data.drop(columns=all_null_columns)

print('\nShape of merged_data after dropping all-null columns:', merged_data.shape)
print('\nNull counts after dropping all-null columns:')
print(merged_data.isnull().sum())

Columns with all null values to be dropped: ['accept_time', 'accept_gps_time', 'delivery_time', 'delivery_gps_time', 'hour', 'weekday', 'time_window_start', 'time_window_end', 'pickup_time', 'pickup_gps_time']

Shape of merged_data after dropping all-null columns: (481415, 16)

Null counts after dropping all-null columns:
order_id                 0
region_id                0
city                     0
courier_id               0
lng                      0
lat                      0
aoi_id                   0
aoi_type                 0
accept_gps_lng      100947
accept_gps_lat      100947
delivery_gps_lng    250000
delivery_gps_lat    250000
ds                       0
is_weekend               0
pickup_gps_lng      280641
pickup_gps_lat      280641
dtype: int64


In [10]:
# Impute remaining numerical missing values (GPS coordinates) with the mean
# Identify columns that still have nulls and are numerical
numerical_cols_with_nulls = merged_data.select_dtypes(include=['number']).columns[merged_data.select_dtypes(include=['number']).isnull().any()].tolist()

print(f'Numerical columns with partial nulls to impute: {numerical_cols_with_nulls}')

for col in numerical_cols_with_nulls:
    mean_val = merged_data[col].mean()
    merged_data[col].fillna(mean_val, inplace=True)
    print(f'Filled missing values in {col} with mean: {mean_val:.4f}')

print('\nShape of merged_data after imputation:', merged_data.shape)
print('\nNull counts after imputation:')
print(merged_data.isnull().sum())

Numerical columns with partial nulls to impute: ['accept_gps_lng', 'accept_gps_lat', 'delivery_gps_lng', 'delivery_gps_lat', 'pickup_gps_lng', 'pickup_gps_lat']
Filled missing values in accept_gps_lng with mean: 119.0247
Filled missing values in accept_gps_lat with mean: 34.0886
Filled missing values in delivery_gps_lng with mean: 118.6436
Filled missing values in delivery_gps_lat with mean: 33.7743
Filled missing values in pickup_gps_lng with mean: 119.1189
Filled missing values in pickup_gps_lat with mean: 34.3886

Shape of merged_data after imputation: (481415, 16)

Null counts after imputation:
order_id            0
region_id           0
city                0
courier_id          0
lng                 0
lat                 0
aoi_id              0
aoi_type            0
accept_gps_lng      0
accept_gps_lat      0
delivery_gps_lng    0
delivery_gps_lat    0
ds                  0
is_weekend          0
pickup_gps_lng      0
pickup_gps_lat      0
dtype: int64


/tmp/ipykernel_2609/152627687.py:9: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  merged_data[col].fillna(mean_val, inplace=True)


### Feature Engineering: Calculating Distances

Now that the data is clean, we can start engineering new features. A common and insightful feature from GPS coordinates is the distance between different points (e.g., pickup and delivery locations). We'll calculate the Euclidean distance between the pickup and delivery GPS coordinates.

In [11]:
# Calculate Euclidean distance between pickup and delivery locations
merged_data['pickup_delivery_distance'] = np.sqrt(
    (merged_data['pickup_gps_lng'] - merged_data['delivery_gps_lng'])**2 +
    (merged_data['pickup_gps_lat'] - merged_data['delivery_gps_lat'])**2
)

# Display the first few rows with the new feature
print(merged_data[['pickup_gps_lng', 'pickup_gps_lat', 'delivery_gps_lng', 'delivery_gps_lat', 'pickup_delivery_distance']].head())
print(f"\nShape of merged_data after adding distance feature: {merged_data.shape}")

   pickup_gps_lng  pickup_gps_lat  delivery_gps_lng  delivery_gps_lat  \
0      119.118933        34.38858         121.35896          37.37456   
1      119.118933        34.38858         121.40707          37.35897   
2      119.118933        34.38858         121.40824          37.35837   
3      119.118933        34.38858         121.52112          37.45253   
4      119.118933        34.38858         121.40067          37.41131   

   pickup_delivery_distance  
0                  3.732800  
1                  3.749505  
2                  3.749744  
3                  3.893365  
4                  3.787244  

Shape of merged_data after adding distance feature: (481415, 17)


All null values have now been handled. The `merged_data` DataFrame no longer contains any missing entries, and its original number of rows has been preserved.

In [12]:
merged_data.isnull().sum()

,0
order_id,0
region_id,0
city,0
courier_id,0
lng,0
lat,0
aoi_id,0
aoi_type,0
accept_gps_lng,0
accept_gps_lat,0


### Feature Engineering: Temporal Features

The `ds` column appears to be a representation of the date. To unlock its full potential for analysis, we'll convert it to a datetime object and then extract common temporal features like the day of the week and day of the month. This can help reveal patterns related to specific days or times.

In [13]:
# Convert 'ds' to datetime objects. Assuming 'ds' represents days since a certain epoch.
# A common epoch is '1970-01-01', but a more recent one might be needed depending on the data origin.
# For now, let's assume it's days since 2000-01-01 to get reasonable dates.
merged_data['date'] = pd.to_datetime(merged_data['ds'], unit='D', origin=pd.Timestamp('2000-01-01'))

# Extract day of the week (Monday=0, Sunday=6)
merged_data['day_of_week'] = merged_data['date'].dt.dayofweek

# Extract day of the month
merged_data['day_of_month'] = merged_data['date'].dt.day

# Display the first few rows with the new temporal features
print(merged_data[['ds', 'date', 'day_of_week', 'day_of_month']].head())
print(f"\nShape of merged_data after adding temporal features: {merged_data.shape}")

    ds       date  day_of_week  day_of_month
0  618 2001-09-10            0            10
1  814 2002-03-25            0            25
2  901 2002-06-20            3            20
3  825 2002-04-05            4             5
4  709 2001-12-10            0            10

Shape of merged_data after adding temporal features: (481415, 20)


In [14]:
# Value counts for day of the week
print("Distribution of Day of Week:")
print(merged_data['day_of_week'].value_counts().sort_index())

print("\nDistribution of Day of Month:")
print(merged_data['day_of_month'].value_counts().sort_index())

Distribution of Day of Week:
day_of_week
0    68923
1    69052
2    67072
3    70210
4    69689
5    67543
6    68926
Name: count, dtype: int64

Distribution of Day of Month:
day_of_month
1     15612
2     15439
3     15524
4     15723
5     15533
6     15955
7     15547
8     15965
9     16442
10    17079
11    17390
12    16468
13    17036
14    16409
15    15566
16    14839
17    15147
18    15338
19    15821
20    15185
21    15759
22    15565
23    13267
24    15270
25    15428
26    14835
27    15397
28    18127
29    14793
30    14882
31    10074
Name: count, dtype: int64


In [15]:
# Check duplicate rows
merged_data.duplicated().sum()


np.int64(0)

In [16]:

# Remove duplicates
merged_data = merged_data.drop_duplicates()

merged_data.duplicated().sum()

np.int64(0)

Handle Inconsistent Data


In [17]:
print("Inspecting the 'city' column for inconsistencies:")
print("The 'city' column is numerical (int64).")
print("Value counts for the 'city' column:")
print(merged_data['city'].value_counts().sort_index())

# If any city IDs appear to be outliers or invalid based on domain knowledge,
# further steps would be needed to handle them (e.g., remapping or removal).
# For now, we are just inspecting the distribution.

Inspecting the 'city' column for inconsistencies:
The 'city' column is numerical (int64).
Value counts for the 'city' column:
city
0    100000
1    100000
2     81415
3    100000
4    100000
Name: count, dtype: int64


In [18]:
# Convert column names to lowercase
merged_data.columns = merged_data.columns.str.lower()

# Replace spaces with underscore
merged_data.columns = merged_data.columns.str.replace(' ', '_')

# View updated column names
print(merged_data.columns)

Index(['order_id', 'region_id', 'city', 'courier_id', 'lng', 'lat', 'aoi_id',
       'aoi_type', 'accept_gps_lng', 'accept_gps_lat', 'delivery_gps_lng',
       'delivery_gps_lat', 'ds', 'is_weekend', 'pickup_gps_lng',
       'pickup_gps_lat', 'pickup_delivery_distance', 'date', 'day_of_week',
       'day_of_month'],
      dtype='object')


Convert Data Types

In [19]:
print('Current data types of merged_data:')
print(merged_data.dtypes)

# Convert 'city' (which is numerical) to string type
# This might be useful if city IDs are to be treated as categorical labels.
merged_data['city'] = merged_data['city'].astype(str)

print('\nData types after converting city to string:')
print(merged_data.dtypes)

Current data types of merged_data:
order_id                             int64
region_id                            int64
city                                 int64
courier_id                           int64
lng                                float64
lat                                float64
aoi_id                               int64
aoi_type                             int64
accept_gps_lng                     float64
accept_gps_lat                     float64
delivery_gps_lng                   float64
delivery_gps_lat                   float64
ds                                   int64
is_weekend                           int64
pickup_gps_lng                     float64
pickup_gps_lat                     float64
pickup_delivery_distance           float64
date                        datetime64[ns]
day_of_week                          int32
day_of_month                         int32
dtype: object

Data types after converting city to string:
order_id                             int64
reg

Label Encoding
Convert categorical values into numbers

In [20]:
from sklearn.preprocessing import LabelEncoder

# Initialize LabelEncoder
le = LabelEncoder()

# Apply Label Encoding to the 'city' column of merged_data
merged_data['city_encoded'] = le.fit_transform(merged_data['city'])

print("Original 'city' column unique values:", merged_data['city'].unique())
print("Encoded 'city_encoded' column unique values:", merged_data['city_encoded'].unique())
print("First 5 rows with original and encoded city:")
print(merged_data[['city', 'city_encoded']].head())

Original 'city' column unique values: ['4' '3' '1' '0' '2']
Encoded 'city_encoded' column unique values: [4 3 1 0 2]
First 5 rows with original and encoded city:
  city  city_encoded
0    4             4
1    4             4
2    4             4
3    4             4
4    4             4


Timestamp Standardization

In [21]:
# The primary DataFrame is 'merged_data', not 'merged_df'.
# The 'date' column in 'merged_data' is already a datetime object (converted from 'ds').
# We will use this existing 'date' column for temporal feature extraction.

# Extract features
# Note: Since the 'date' column was derived from 'ds' (days since epoch) and does not contain
# a time component, extracting 'hour' will result in all zeros.
merged_data['hour'] = merged_data['date'].dt.hour
merged_data['day'] = merged_data['date'].dt.day_name()

# The 'is_weekend' feature has already been calculated in a previous step (cell Cn6h2zR85tIA).


Location Data Normalization


In [22]:
merged_data['lat'] = merged_data['lat'].astype(float)
merged_data['lng'] = merged_data['lng'].astype(float)

Merging Datasets

In [23]:
merged_data = pd.merge(delivery_data, pickup_data, on='order_id', how='inner')

Initial Feature Engineering

In [24]:
# First, ensure the 'date' column exists by converting from 'ds_x'
# This step is necessary because the merged_data was overwritten by the pd.merge operation,
# losing previously created columns like 'date'.
merged_data['date'] = pd.to_datetime(merged_data['ds_x'], unit='D', origin=pd.Timestamp('2000-01-01'))

# Now, calculate 'is_weekend' using the newly created 'date' column
merged_data['is_weekend'] = merged_data['date'].dt.weekday >= 5

In [25]:
display(merged_data.head())

,order_id,region_id_x,city_x,courier_id_x,lng_x,lat_x,aoi_id_x,aoi_type_x,accept_time_x,accept_gps_time_x,...,pickup_gps_lat,accept_gps_time_y,accept_gps_lng_y,accept_gps_lat_y,ds_y,hour_y,weekday_y,is_weekend_y,date,is_weekend
0,2374864,86,4,1586,121.52549,37.41730,453,14,NaN,NaN,...,NaN,NaN,121.73044,31.20856,1006,NaN,NaN,0,2001-12-04,False
1,4128020,86,4,1586,121.52516,37.42048,453,14,NaN,NaN,...,43.86189,NaN,126.54473,43.85267,710,NaN,NaN,0,2002-07-05,False
2,1694074,86,4,1586,121.52556,37.42046,453,14,NaN,NaN,...,NaN,NaN,120.03345,30.37764,1020,NaN,NaN,0,2001-12-17,False
3,4460994,86,4,1693,121.40562,37.48776,639,1,NaN,NaN,...,43.72486,NaN,127.33801,43.72526,725,NaN,NaN,0,2001-12-22,True
4,531210,86,4,1693,121.40862,37.48384,639,1,NaN,NaN,...,NaN,NaN,NaN,NaN,611,NaN,NaN,0,2001-12-28,False


### Comprehensive Summary of `merged_data`

In [26]:
# 1. General information about columns and data types
print('--- DataFrame Info ---')
print(merged_data.info())

# 2. Check for any remaining null values
print('\n--- Null Value Counts ---')
print(merged_data.isnull().sum())

# 3. Descriptive statistics for numerical columns
print('\n--- Numerical Summary ---')
display(merged_data.describe())

# 4. Preview the first few rows
print('\n--- Data Preview ---')
display(merged_data.head())

--- DataFrame Info ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9170 entries, 0 to 9169
Data columns (total 43 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   order_id           9170 non-null   int64         
 1   region_id_x        9170 non-null   int64         
 2   city_x             9170 non-null   int64         
 3   courier_id_x       9170 non-null   int64         
 4   lng_x              9170 non-null   float64       
 5   lat_x              9170 non-null   float64       
 6   aoi_id_x           9170 non-null   int64         
 7   aoi_type_x         9170 non-null   int64         
 8   accept_time_x      0 non-null      float64       
 9   accept_gps_time_x  0 non-null      float64       
 10  accept_gps_lng_x   9170 non-null   float64       
 11  accept_gps_lat_x   9170 non-null   float64       
 12  delivery_time      0 non-null      float64       
 13  delivery_gps_time  0 non-null      float

,order_id,region_id_x,city_x,courier_id_x,lng_x,lat_x,aoi_id_x,aoi_type_x,accept_time_x,accept_gps_time_x,...,pickup_gps_lng,pickup_gps_lat,accept_gps_time_y,accept_gps_lng_y,accept_gps_lat_y,ds_y,hour_y,weekday_y,is_weekend_y,date
count,9.170000e+03,9170.000000,9170.000000,9170.000000,9170.000000,9170.000000,9170.000000,9170.000000,0.0,0.0,...,7411.000000,7411.000000,0.0,5477.000000,5477.000000,9170.000000,0.0,0.0,9170.0,9170
mean,2.275149e+06,34.905234,2.000872,2002.261941,118.559898,33.731878,25818.314613,4.480698,NaN,NaN,...,119.110563,34.448744,NaN,119.620509,34.631154,780.384406,NaN,NaN,0.0,2002-03-11 00:46:47.764449280
min,7.510000e+02,0.000000,0.000000,3.000000,106.340300,29.574340,9.000000,0.000000,NaN,NaN,...,106.322340,29.449250,NaN,106.266950,29.153370,501.000000,NaN,NaN,0.0,2001-05-16 00:00:00
25%,1.161351e+06,1.000000,1.000000,783.000000,120.173442,30.261280,10088.500000,1.000000,NaN,NaN,...,120.003475,30.352790,NaN,120.013590,30.360390,622.000000,NaN,NaN,0.0,2001-09-21 00:00:00
50%,2.289876e+06,15.000000,2.000000,1789.000000,121.249475,31.120610,22762.000000,1.000000,NaN,NaN,...,121.196290,31.187000,NaN,121.226410,31.185930,805.000000,NaN,NaN,0.0,2002-03-26 00:00:00
75%,3.373106e+06,86.000000,3.000000,3156.000000,121.520218,37.448925,40936.000000,12.000000,NaN,NaN,...,121.701610,37.535360,NaN,121.709730,37.535740,919.000000,NaN,NaN,0.0,2002-07-13 00:00:00
max,4.514015e+06,165.000000,4.000000,4874.000000,126.675480,44.122390,60110.000000,15.000000,NaN,NaN,...,127.352050,44.426550,NaN,127.348310,44.426110,1031.000000,NaN,NaN,0.0,2002-10-28 00:00:00
std,1.290473e+06,44.313895,1.479934,1438.181115,6.607493,4.869558,17650.931989,5.650422,NaN,NaN,...,6.936441,5.480377,NaN,6.668883,5.603873,164.401074,NaN,NaN,0.0,NaN



--- Data Preview ---


,order_id,region_id_x,city_x,courier_id_x,lng_x,lat_x,aoi_id_x,aoi_type_x,accept_time_x,accept_gps_time_x,...,pickup_gps_lat,accept_gps_time_y,accept_gps_lng_y,accept_gps_lat_y,ds_y,hour_y,weekday_y,is_weekend_y,date,is_weekend
0,2374864,86,4,1586,121.52549,37.41730,453,14,NaN,NaN,...,NaN,NaN,121.73044,31.20856,1006,NaN,NaN,0,2001-12-04,False
1,4128020,86,4,1586,121.52516,37.42048,453,14,NaN,NaN,...,43.86189,NaN,126.54473,43.85267,710,NaN,NaN,0,2002-07-05,False
2,1694074,86,4,1586,121.52556,37.42046,453,14,NaN,NaN,...,NaN,NaN,120.03345,30.37764,1020,NaN,NaN,0,2001-12-17,False
3,4460994,86,4,1693,121.40562,37.48776,639,1,NaN,NaN,...,43.72486,NaN,127.33801,43.72526,725,NaN,NaN,0,2001-12-22,True
4,531210,86,4,1693,121.40862,37.48384,639,1,NaN,NaN,...,NaN,NaN,NaN,NaN,611,NaN,NaN,0,2001-12-28,False


### Finalizing Data Cleaning

Based on the summary, we have several columns that are 100% null and some coordinate columns with partial nulls. We will now drop the useless columns and impute the remaining values.

In [27]:
# 1. Drop columns that are entirely null
null_cols_to_drop = merged_data.columns[merged_data.isnull().all()].tolist()
merged_data.drop(columns=null_cols_to_drop, inplace=True)
print(f'Dropped {len(null_cols_to_drop)} entirely null columns.')

# 2. Impute partial nulls in GPS columns with their mean
# Note: This affects columns like pickup_gps_lng and accept_gps_lng_y
gps_cols_to_impute = merged_data.columns[merged_data.isnull().any()].tolist()
for col in gps_cols_to_impute:
    merged_data[col] = merged_data[col].fillna(merged_data[col].mean())

print(f'Imputed remaining nulls in: {gps_cols_to_impute}')
print('Total nulls now:', merged_data.isnull().sum().sum())

Dropped 14 entirely null columns.
Imputed remaining nulls in: ['pickup_gps_lng', 'pickup_gps_lat', 'accept_gps_lng_y', 'accept_gps_lat_y']
Total nulls now: 0


### Outlier Detection

We will check if any GPS coordinates are extreme outliers, which could indicate bad data entry.

In [28]:
# Simple check for latitude/longitude outliers
# Standard ranges for this dataset seem to be Lat [29, 45] and Lng [106, 127]
def check_outliers(df, col, min_val, max_val):
    outliers = df[(df[col] < min_val) | (df[col] > max_val)]
    print(f'Outliers in {col}: {len(outliers)}')
    return outliers

# Checking a few key coordinate columns
check_outliers(merged_data, 'lat_x', 20, 50)
check_outliers(merged_data, 'lng_x', 100, 130)



Outliers in lat_x: 0
Outliers in lng_x: 0


,order_id,region_id_x,city_x,courier_id_x,lng_x,lat_x,aoi_id_x,aoi_type_x,accept_gps_lng_x,accept_gps_lat_x,...,aoi_id_y,aoi_type_y,pickup_gps_lng,pickup_gps_lat,accept_gps_lng_y,accept_gps_lat_y,ds_y,is_weekend_y,date,is_weekend
